# v5 Local Experiment — MiniCPM-V 4.6 QLoRA (RTX 5060 Ti / CUDA 12.8)

이 버전은 `(260902)_baseline_desktop5060ti_offline.ipynb`의 **검증된 로컬 환경 방식**을 기준으로 수정했습니다.

- Python 3.13.x
- PyTorch `2.11.0+cu128`
- torchvision `0.26.0+cu128`
- CUDA 12.8 / RTX 5060 Ti
- Transformers `5.7.0`
- 최신 bitsandbytes (CUDA 12.8 / Blackwell 지원)
- Windows에서는 FlashAttention을 강제로 설치하지 않고 PyTorch SDPA 사용

먼저 **환경 설치 셀 실행 → VSCode 커널 재시작 → 환경 검증 셀부터 다시 실행**하세요.


In [3]:
from pathlib import Path
import os, json, re, random, shutil, subprocess, sys, math
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
LIB_DIR = ROOT / "downloads" / "libs"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# 오프라인이면 다운로드해 둔 모델 경로를 사용하세요.
LOCAL_MODEL_DIR = ROOT / "downloads" / "models" / "MiniCPM-V-4.6"
OFFLINE = False
MODEL_ID = str(LOCAL_MODEL_DIR) if OFFLINE else "openbmb/MiniCPM-V-4.6"

SEED = 42
VAL_FRAC = 0.10
TRAIN_LIMIT = 300        # 먼저 smoke test. 정상 완료 후 None으로 변경

# 실험 하나만 바꿔서 재실행합니다.
EXPERIMENT = "A_backbone_16x"
EXPERIMENTS = {
    "A_backbone_16x": dict(
        downsample_mode="16x",
        freeze_vision_tower=True,
        freeze_multi_modal_projector=True,
    ),
    "B_resolution_4x": dict(
        downsample_mode="4x",
        freeze_vision_tower=True,
        freeze_multi_modal_projector=True,
    ),
    "C_projector_4x": dict(
        downsample_mode="4x",
        freeze_vision_tower=True,
        freeze_multi_modal_projector=False,
    ),
}
CFG = EXPERIMENTS[EXPERIMENT]
RUN_DIR = OUTPUT_DIR / f"v5_minicpm46_{EXPERIMENT}"
ADAPTER_DIR = RUN_DIR / "adapter"
MERGED_DIR = RUN_DIR / "merged"
LF_DATA_DIR = RUN_DIR / "lf_data"
for p in [RUN_DIR, LF_DATA_DIR]: p.mkdir(parents=True, exist_ok=True)

if OFFLINE:
    assert LOCAL_MODEL_DIR.exists(), f"모델이 없습니다: {LOCAL_MODEL_DIR}"

print("experiment:", EXPERIMENT)
print("model     :", MODEL_ID)
print("run dir   :", RUN_DIR)
print("settings  :", CFG)


experiment: A_backbone_16x
model     : openbmb/MiniCPM-V-4.6
run dir   : c:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x
settings  : {'downsample_mode': '16x', 'freeze_vision_tower': True, 'freeze_multi_modal_projector': True}


## 0. 환경 설치 — 기존 RTX 5060 Ti 베이스라인 방식 유지

기존 베이스라인에서 정상 동작한 `torch==2.11.0+cu128`을 **먼저 고정**합니다.
그 다음 MiniCPM-V 4.6에 필요한 패키지를 설치합니다.

### 중요
- `transformers[torch]`를 설치하지 않습니다. CPU/다른 CUDA torch로 덮어쓰는 상황을 피합니다.
- Windows에서는 `flash-attn`을 설치하지 않습니다. 아래 학습 설정은 `sdpa`를 사용합니다.
- 오프라인 설치라면 `downloads/libs`에 아래 버전 wheel들이 미리 있어야 합니다.
- 처음 환경을 고친 직후에는 **반드시 커널을 재시작**합니다.


In [4]:
# ===== 의존성 설치 =====
# 기존 (260902) RTX 5060 Ti 베이스라인과 같은 CUDA 12.8 PyTorch를 유지합니다.

import sys, subprocess
from pathlib import Path

# True: 기존 baseline처럼 downloads/libs의 wheel만 사용
# False: 인터넷에서 설치하되 torch는 반드시 CUDA 12.8 인덱스에서 설치
INSTALL_FROM_LOCAL_WHEELS = False

def run_pip(args):
    cmd = [sys.executable, "-m", "pip", *args]
    print("\n>", " ".join(map(str, cmd)))
    subprocess.check_call(cmd)

# -------------------------------------------------------
# 1) CUDA PyTorch를 가장 먼저 고정
# -------------------------------------------------------
TORCH_PKGS = [
    "torch==2.11.0+cu128",
    "torchvision==0.26.0+cu128",
    "torchaudio==2.11.0+cu128",
]

if INSTALL_FROM_LOCAL_WHEELS:
    assert LIB_DIR.exists(), f"wheel 폴더가 없습니다: {LIB_DIR}"
    run_pip([
        "install",
        "--no-index",
        "--find-links", str(LIB_DIR),
        *TORCH_PKGS,
    ])
else:
    run_pip([
        "install",
        "--index-url", "https://download.pytorch.org/whl/cu128",
        *TORCH_PKGS,
    ])

# -------------------------------------------------------
# 2) MiniCPM-V 4.6 + QLoRA 핵심 의존성
#
# accelerate 1.11.0:
#   현재 LLaMA-Factory의 지원 범위(<=1.11.0)에 맞춤
#
# bitsandbytes >=0.48.2:
#   Windows + CUDA 12.8 + Blackwell(sm120) wheel 지원
# -------------------------------------------------------
CORE_PKGS = [
    "transformers==5.7.0",
    "accelerate==1.11.0",
    "peft==0.18.1",
    "trl==0.24.0",
    "datasets>=2.16.0,<=4.0.0",
    "bitsandbytes>=0.48.2",
    "torchdata>=0.10.0,<=0.11.0",
    "av>=10.0.0,<=16.0.0",
    "pillow",
    "pandas",
    "pyyaml",
    "einops",
    "sentencepiece",
    "safetensors",
    "tokenizers",
    "packaging",
]

if INSTALL_FROM_LOCAL_WHEELS:
    run_pip([
        "install",
        "--no-index",
        "--find-links", str(LIB_DIR),
        "--upgrade-strategy", "only-if-needed",
        *CORE_PKGS,
    ])
else:
    run_pip([
        "install",
        "--upgrade-strategy", "only-if-needed",
        *CORE_PKGS,
    ])

# -------------------------------------------------------
# 3) LLaMA-Factory
# -------------------------------------------------------
if INSTALL_FROM_LOCAL_WHEELS:
    # downloads/libs에 최신 LLaMA-Factory wheel을 미리 받아둔 경우
    lf_wheels = sorted(LIB_DIR.glob("llamafactory-*.whl"))
    if not lf_wheels:
        raise FileNotFoundError(
            "downloads/libs에 최신 llamafactory wheel이 없습니다.\n"
            "처음 준비할 때 INSTALL_FROM_LOCAL_WHEELS=False로 설치하거나,\n"
            "최신 LLaMA-Factory wheel을 downloads/libs에 추가하세요."
        )
    run_pip([
        "install",
        "--no-index",
        "--find-links", str(LIB_DIR),
        "--upgrade-strategy", "only-if-needed",
        str(lf_wheels[-1]),
    ])
else:
    # MiniCPM-V 4.6 지원이 들어간 최신 소스 사용
    run_pip([
        "install",
        "--upgrade-strategy", "only-if-needed",
        "git+https://github.com/hiyouga/LLaMA-Factory.git",
    ])

print("\n설치 완료.")
print("중요: 지금 VSCode에서 Restart Kernel 한 뒤, 다음 '환경 검증' 셀부터 다시 실행하세요.")



> c:\SSAFY\AIChallenge\baseline\Scripts\python.exe -m pip install --index-url https://download.pytorch.org/whl/cu128 torch==2.11.0+cu128 torchvision==0.26.0+cu128 torchaudio==2.11.0+cu128

> c:\SSAFY\AIChallenge\baseline\Scripts\python.exe -m pip install --upgrade-strategy only-if-needed transformers==5.7.0 accelerate==1.11.0 peft==0.18.1 trl==0.24.0 datasets>=2.16.0,<=4.0.0 bitsandbytes>=0.48.2 torchdata>=0.10.0,<=0.11.0 av>=10.0.0,<=16.0.0 pillow pandas pyyaml einops sentencepiece safetensors tokenizers packaging

> c:\SSAFY\AIChallenge\baseline\Scripts\python.exe -m pip install --upgrade-strategy only-if-needed git+https://github.com/hiyouga/LLaMA-Factory.git

설치 완료.
중요: 지금 VSCode에서 Restart Kernel 한 뒤, 다음 '환경 검증' 셀부터 다시 실행하세요.


In [5]:
# ===== 환경 검증 =====
# 의존성 설치 후 반드시 커널을 재시작하고 이 셀을 실행하세요.

import sys, shutil
import torch
import torchvision
import transformers
import accelerate
import peft
import bitsandbytes as bnb
from packaging.version import Version

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("torch cuda  :", torch.version.cuda)
print("torchvision :", torchvision.__version__)
print("transformers:", transformers.__version__)
print("accelerate  :", accelerate.__version__)
print("peft        :", peft.__version__)
print("bitsandbytes:", bnb.__version__)
print("cuda usable :", torch.cuda.is_available())

assert "+cu128" in torch.__version__, (
    f"CUDA 12.8 torch가 아닙니다: {torch.__version__}\n"
    "위 설치 셀에서 torch==2.11.0+cu128을 먼저 다시 설치하세요."
)
assert torch.version.cuda == "12.8", f"PyTorch CUDA build가 12.8이 아닙니다: {torch.version.cuda}"
assert torch.cuda.is_available(), (
    "PyTorch가 GPU를 인식하지 못합니다. "
    "커널을 재시작했는지, 그리고 위 CUDA 12.8 torch 설치가 성공했는지 확인하세요."
)

print("gpu         :", torch.cuda.get_device_name(0))
print("capability  :", torch.cuda.get_device_capability(0))

assert Version(transformers.__version__) >= Version("5.7.0")
assert Version(bnb.__version__) >= Version("0.48.2")
assert shutil.which("llamafactory-cli"), (
    "llamafactory-cli가 없습니다. "
    "환경 설치 셀의 LLaMA-Factory 설치 부분을 확인하세요."
)

print("\n환경 OK")


python      : 3.12.10
torch       : 2.11.0+cu128
torch cuda  : 12.8
torchvision : 0.26.0+cu128
transformers: 5.7.0
accelerate  : 1.11.0
peft        : 0.18.1
bitsandbytes: 0.50.2
cuda usable : True
gpu         : NVIDIA GeForce RTX 5060 Ti
capability  : (12, 0)

환경 OK


## 1. v4와 동일한 group split 생성


In [6]:
random.seed(SEED)
np.random.seed(SEED)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

if TRAIN_LIMIT is not None:
    train_df = train_df.sample(n=min(TRAIN_LIMIT, len(train_df)), random_state=SEED).reset_index(drop=True)

def normalize_question(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())

def make_group_split(df, val_frac=0.10, seed=42, trials=100):
    groups = {}
    for idx, q in enumerate(df["question"].map(normalize_question)):
        groups.setdefault(q, []).append(idx)

    group_items = list(groups.items())
    target_n = int(round(len(df) * val_frac))
    overall = df["answer"].astype(str).str.lower().value_counts(normalize=True).reindex(list("abcd"), fill_value=0.0)

    best = None
    for t in range(trials):
        rng = random.Random(seed + t)
        items = group_items.copy()
        rng.shuffle(items)
        val_idx = []
        for _, idxs in items:
            if len(val_idx) >= target_n:
                break
            val_idx.extend(idxs)
        val_idx = sorted(set(val_idx))
        val_prop = df.iloc[val_idx]["answer"].astype(str).str.lower().value_counts(normalize=True).reindex(list("abcd"), fill_value=0.0)
        score = float((val_prop - overall).abs().sum()) + abs(len(val_idx) - target_n) / len(df)
        if best is None or score < best[0]:
            best = (score, val_idx)

    val_idx = set(best[1])
    train_idx = [i for i in range(len(df)) if i not in val_idx]
    return df.iloc[train_idx].reset_index(drop=True), df.iloc[sorted(val_idx)].reset_index(drop=True)

train_subset, valid_subset = make_group_split(train_df, VAL_FRAC, SEED)
print("train/valid:", len(train_subset), len(valid_subset))
print(valid_subset["answer"].astype(str).str.lower().value_counts(normalize=True).sort_index())

# 다른 모델/노트북과 validation 행을 맞추는 키
valid_keys = valid_subset[[c for c in ["id", "path", "question", "answer"] if c in valid_subset.columns]].copy()
valid_keys.to_csv(RUN_DIR / "valid_keys.csv", index=False)


train/valid: 269 31
answer
a    0.225806
b    0.258065
c    0.193548
d    0.322581
Name: proportion, dtype: float64


## 2. LLaMA-Factory용 multimodal SFT 데이터 생성


In [7]:
SYSTEM_INSTRUCT = (
    "이미지를 보고 객관식 질문에 답하세요. "
    "필요하면 이미지 속 글자, 숫자, 표지판, 가격, 상호명 등 세부 정보를 주의 깊게 읽으세요. "
    "최종 답은 반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요."
)

def build_mc_prompt(row):
    return (
        f"질문: {row['question']}\n"
        f"(a) {row['a']}\n"
        f"(b) {row['b']}\n"
        f"(c) {row['c']}\n"
        f"(d) {row['d']}\n"
        "정답:"
    )

def to_lf_records(df, with_answer=True):
    records = []
    for _, row in df.iterrows():
        img_path = (DATA_DIR / str(row["path"])).resolve()
        assert img_path.exists(), img_path
        msgs = [
            {"role": "system", "content": SYSTEM_INSTRUCT},
            {"role": "user", "content": "<image>\n" + build_mc_prompt(row)},
        ]
        if with_answer:
            msgs.append({"role": "assistant", "content": str(row["answer"]).strip().lower()})
        records.append({"messages": msgs, "images": [str(img_path)]})
    return records

train_json = LF_DATA_DIR / "kaggle_v5_train.json"
valid_json = LF_DATA_DIR / "kaggle_v5_valid.json"
train_json.write_text(json.dumps(to_lf_records(train_subset), ensure_ascii=False, indent=2), encoding="utf-8")
valid_json.write_text(json.dumps(to_lf_records(valid_subset), ensure_ascii=False, indent=2), encoding="utf-8")

info = {
    "kaggle_v5_train": {
        "file_name": train_json.name,
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": {
            "role_tag": "role", "content_tag": "content",
            "user_tag": "user", "assistant_tag": "assistant", "system_tag": "system"
        },
    },
    "kaggle_v5_valid": {
        "file_name": valid_json.name,
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": {
            "role_tag": "role", "content_tag": "content",
            "user_tag": "user", "assistant_tag": "assistant", "system_tag": "system"
        },
    },
}
(LF_DATA_DIR / "dataset_info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", train_json, valid_json, LF_DATA_DIR / "dataset_info.json", sep="\n")


saved:
c:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\lf_data\kaggle_v5_train.json
c:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\lf_data\kaggle_v5_valid.json
c:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\lf_data\dataset_info.json


## 3. 4-bit QLoRA 설정 생성


### Windows/RTX 5060 Ti용 학습 설정 보정

- 첫 실행은 `TRAIN_LIMIT=300`으로 smoke test합니다.
- A 실험은 language-only LoRA이므로 `lora_target=q_proj,v_proj`만 적용합니다.
- Windows에서는 `preprocessing_num_workers=1`, `dataloader_num_workers=0`으로 시작합니다.
- `flash_attn=auto`로 두어 설치되지 않은 FlashAttention을 강제하지 않습니다.
- 학습 실행 셀은 LLaMA-Factory의 stdout/stderr를 모두 실시간 출력하고 로그 파일로 저장합니다.


In [8]:
import yaml

train_cfg = {
    # model
    "model_name_or_path": MODEL_ID,
    "trust_remote_code": True,
    "flash_attn": "auto",

    # method
    "stage": "sft",
    "do_train": True,
    "finetuning_type": "lora",
    "lora_target": "q_proj,v_proj",
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "freeze_vision_tower": CFG["freeze_vision_tower"],
    "freeze_multi_modal_projector": CFG["freeze_multi_modal_projector"],

    # QLoRA
    "quantization_bit": 4,
    "quantization_method": "bnb",
    "double_quantization": True,

    # data
    "dataset_dir": str(LF_DATA_DIR.resolve()),
    "dataset": "kaggle_v5_train",
    "template": "minicpm_v_4_6",
    "cutoff_len": 4096,
    "packing": False,
    "overwrite_cache": True,
    "preprocessing_num_workers": 1,
    "dataloader_num_workers": 0,

    # train — v4와 최대한 동일하게
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 5e-5,
    "num_train_epochs": 1.0,
    "lr_scheduler_type": "linear",
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "bf16": bool(torch.cuda.is_bf16_supported()),
    "fp16": not bool(torch.cuda.is_bf16_supported()),

    # output
    "output_dir": str(ADAPTER_DIR.resolve()),
    "logging_steps": 5,
    "save_strategy": "epoch",
    "save_total_limit": 1,
    "plot_loss": True,
    "overwrite_output_dir": True,
    "report_to": "none",
}

TRAIN_YAML = RUN_DIR / "train.yaml"
TRAIN_YAML.write_text(yaml.safe_dump(train_cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(TRAIN_YAML.read_text(encoding="utf-8"))


model_name_or_path: openbmb/MiniCPM-V-4.6
trust_remote_code: true
flash_attn: auto
stage: sft
do_train: true
finetuning_type: lora
lora_target: q_proj,v_proj
lora_rank: 8
lora_alpha: 16
lora_dropout: 0.05
freeze_vision_tower: true
freeze_multi_modal_projector: true
quantization_bit: 4
quantization_method: bnb
double_quantization: true
dataset_dir: C:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\lf_data
dataset: kaggle_v5_train
template: minicpm_v_4_6
cutoff_len: 4096
packing: false
overwrite_cache: true
preprocessing_num_workers: 1
dataloader_num_workers: 0
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 5.0e-05
num_train_epochs: 1.0
lr_scheduler_type: linear
warmup_ratio: 0.05
weight_decay: 0.01
max_grad_norm: 1.0
bf16: true
fp16: false
output_dir: C:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\adapter
logging_steps: 5
save_strategy: epoch
save_total_limit: 1
plot_loss: true
overwrite_output_dir: true
report_to: none



## 4. 학습

MiniCPM-V 4.6은 `DOWNSAMPLE_MODE=16x/4x`를 사용합니다. `16x`는 빠르고, `4x`는 visual token이 더 많아 작은 글자/OCR에 유리한지 검증하는 실험입니다.


## MiniCPM-V 4.6 × LLaMA-Factory 4bit 호환 패치

MiniCPM-V 4.6의 `model.merger`는 경우에 따라 `list`를 반환하지만, 현재 LLaMA-Factory의 `autocast_projector_dtype()` hook은 projector 출력이 Tensor라고 가정하고 `.to()`를 직접 호출합니다.

아래 셀은 **LLaMA-Factory의 해당 hook만 최소 수정**하여 Tensor / list / tuple / dict 출력을 모두 안전하게 dtype 변환합니다. 원본 파일은 `.minicpm46_backup`으로 한 번 백업합니다.

패치 후 별도 커널 재시작은 필요 없습니다. 학습 subprocess는 수정된 파일을 새로 import합니다.


In [9]:
# ===== MiniCPM-V 4.6 + 4bit LLaMA-Factory compatibility patch =====
from pathlib import Path
import llamafactory.model.model_utils.visual as lf_visual

visual_py = Path(lf_visual.__file__).resolve()
backup_py = visual_py.with_suffix(visual_py.suffix + ".minicpm46_backup")

text = visual_py.read_text(encoding="utf-8")

old = """        return output.to(model_args.compute_dtype)
"""

new = """        def _cast_projector_output(value):
            if torch.is_tensor(value):
                return value.to(model_args.compute_dtype)
            if isinstance(value, list):
                return [_cast_projector_output(item) for item in value]
            if isinstance(value, tuple):
                return tuple(_cast_projector_output(item) for item in value)
            if isinstance(value, dict):
                return {key: _cast_projector_output(item) for key, item in value.items()}
            return value

        return _cast_projector_output(output)
"""

if "_cast_projector_output" in text:
    print("이미 MiniCPM-V 4.6 호환 패치가 적용되어 있습니다.")
else:
    if old not in text:
        raise RuntimeError(
            "LLaMA-Factory visual.py 구조가 예상과 다릅니다.\n"
            f"파일: {visual_py}\n"
            "visual.py의 autocast_projector_dtype 구현을 다시 확인해야 합니다."
        )

    if not backup_py.exists():
        backup_py.write_text(text, encoding="utf-8")
        print("원본 백업:", backup_py)

    patched = text.replace(old, new, 1)
    visual_py.write_text(patched, encoding="utf-8")
    print("패치 적용:", visual_py)

# 실제 수정 확인
verify = visual_py.read_text(encoding="utf-8")
assert "_cast_projector_output" in verify
print("MiniCPM-V 4.6 projector-output patch OK")


원본 백업: C:\SSAFY\AIChallenge\baseline\Lib\site-packages\llamafactory\model\model_utils\visual.py.minicpm46_backup
패치 적용: C:\SSAFY\AIChallenge\baseline\Lib\site-packages\llamafactory\model\model_utils\visual.py
MiniCPM-V 4.6 projector-output patch OK


In [10]:
RUN_TRAIN = True

env = os.environ.copy()

# MiniCPM-V 4.6 visual compression experiment
env["DOWNSAMPLE_MODE"] = CFG["downsample_mode"]

# Official MiniCPM-V 4.6 LLaMA-Factory launcher guidance:
# do not use legacy v1 launcher.
env.pop("USE_V1", None)
env["DISABLE_VERSION_CHECK"] = "1"

# Windows console/subprocess text handling
env["PYTHONUTF8"] = "1"
env["PYTHONIOENCODING"] = "utf-8"

if OFFLINE:
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
else:
    # 이전 셀/환경에 남아 있으면 온라인 모델 다운로드를 막을 수 있으므로 제거
    env.pop("HF_HUB_OFFLINE", None)
    env.pop("TRANSFORMERS_OFFLINE", None)

# 설치된 LLaMA-Factory 위치/버전 확인
print("llamafactory-cli:", shutil.which("llamafactory-cli"))
ver = subprocess.run(
    ["llamafactory-cli", "version"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print(ver.stdout)

cmd = ["llamafactory-cli", "train", str(TRAIN_YAML)]
print("COMMAND        :", " ".join(cmd))
print("DOWNSAMPLE_MODE:", env["DOWNSAMPLE_MODE"])
print("MODEL          :", MODEL_ID)
print("DATASET_DIR    :", LF_DATA_DIR.resolve())
print("OUTPUT         :", ADAPTER_DIR.resolve())

if RUN_TRAIN:
    # check=True만 쓰면 Jupyter에서 실제 내부 traceback을 놓치기 쉬워서
    # stdout/stderr를 합쳐 실시간으로 그대로 출력합니다.
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    log_lines = []
    assert proc.stdout is not None

    for line in proc.stdout:
        print(line, end="")
        log_lines.append(line)

    return_code = proc.wait()

    # 로그도 파일로 남김
    log_path = RUN_DIR / "llamafactory_train.log"
    log_path.write_text("".join(log_lines), encoding="utf-8")

    if return_code != 0:
        print("\n" + "=" * 80)
        print("LLaMA-Factory training failed.")
        print("return code:", return_code)
        print("full log   :", log_path)
        print("=" * 80)
        print("\n===== LAST 100 LOG LINES =====\n")
        print("".join(log_lines[-100:]))
        raise RuntimeError(
            f"LLaMA-Factory가 종료 코드 {return_code}로 실패했습니다. "
            f"위에 출력된 마지막 traceback 또는 {log_path}를 확인하세요."
        )

    print("\n학습 완료:", ADAPTER_DIR)


llamafactory-cli: c:\SSAFY\AIChallenge\baseline\Scripts\llamafactory-cli.EXE
----------------------------------------------------------
| Welcome to LLaMA Factory, version 0.9.6.dev0           |
|                                                        |
| Project page: https://github.com/hiyouga/LLaMA-Factory |
----------------------------------------------------------

COMMAND        : llamafactory-cli train c:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\train.yaml
DOWNSAMPLE_MODE: 16x
MODEL          : openbmb/MiniCPM-V-4.6
DATASET_DIR    : C:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\lf_data
OUTPUT         : C:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\adapter
[WARNING|2026-09-21 16:54:05] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[WARNING|2026-09-21 16:54:06] llamafactory.hparams.parser:149 >> We

## 5. LoRA merge — 평가는 merged BF16 모델로 단순화


In [11]:
merge_cfg = {
    "model_name_or_path": MODEL_ID,
    "adapter_name_or_path": str(ADAPTER_DIR.resolve()),
    "template": "minicpm_v_4_6",
    "finetuning_type": "lora",
    "trust_remote_code": True,
    "export_dir": str(MERGED_DIR.resolve()),
    "export_size": 2,
    "export_device": "auto",
    "export_legacy_format": False,
}
MERGE_YAML = RUN_DIR / "merge.yaml"
MERGE_YAML.write_text(yaml.safe_dump(merge_cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(MERGE_YAML.read_text(encoding="utf-8"))

RUN_MERGE = True
if RUN_MERGE:
    # 중요: merge 시 quantization_bit를 넣지 않습니다.
    subprocess.run(["llamafactory-cli", "export", str(MERGE_YAML)], env=env, check=True)


model_name_or_path: openbmb/MiniCPM-V-4.6
adapter_name_or_path: C:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\adapter
template: minicpm_v_4_6
finetuning_type: lora
trust_remote_code: true
export_dir: C:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\merged
export_size: 2
export_device: auto
export_legacy_format: false



## 6. Validation a/b/c/d score 저장

`generate()` 문자열 파싱 대신 assistant 첫 토큰의 `a/b/c/d` logits을 비교합니다. 시작 전에 몇 샘플을 generate해 **첫 출력이 실제로 a/b/c/d인지 sanity check**합니다.


In [12]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from tqdm.auto import tqdm

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
processor = AutoProcessor.from_pretrained(
    str(MERGED_DIR),
    local_files_only=OFFLINE,
)
model = AutoModelForImageTextToText.from_pretrained(
    str(MERGED_DIR),
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
    local_files_only=OFFLINE,
)
model.eval()
MODEL_DEVICE = next(model.parameters()).device
print("model device:", MODEL_DEVICE)

CHOICES = ["a", "b", "c", "d"]
choice_ids = []
for c in CHOICES:
    ids = processor.tokenizer.encode(c, add_special_tokens=False)
    print(c, ids)
    if len(ids) != 1:
        raise RuntimeError(f"{c!r}가 단일 토큰이 아닙니다. 이 경우 sequence scoring으로 바꾸세요: {ids}")
    choice_ids.append(ids[0])
choice_ids = torch.tensor(choice_ids, device=MODEL_DEVICE)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 779/779 [00:00<00:00, 811.30it/s] 


model device: cuda:0
a [64]
b [65]
c [66]
d [67]


In [13]:
def build_messages(row):
    img_path = (DATA_DIR / str(row["path"])).resolve()
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "path": str(img_path)},
            {"type": "text", "text": build_mc_prompt(row)},
        ]},
    ]

@torch.inference_mode()
def smoke_generate(df, n=3):
    for _, row in df.head(n).iterrows():
        inputs = processor.apply_chat_template(
            build_messages(row), tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt",
            downsample_mode=CFG["downsample_mode"], max_slice_nums=36,
        ).to(MODEL_DEVICE)
        out = model.generate(
            **inputs,
            downsample_mode=CFG["downsample_mode"],
            max_new_tokens=8,
            do_sample=False,
        )
        new_ids = out[0, inputs["input_ids"].shape[1]:]
        text = processor.decode(new_ids, skip_special_tokens=True).strip()
        print("gold=", str(row["answer"]).lower(), "generated=", repr(text))

smoke_generate(valid_subset, 3)
print("첫 글자가 a/b/c/d가 아니면 아래 logit scoring을 그대로 쓰지 말고 prompt/template부터 확인하세요.")


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


gold= d generated= 'Looking at the price tag for the "'


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


gold= c generated= 'd'
gold= a generated= 'c'
첫 글자가 a/b/c/d가 아니면 아래 logit scoring을 그대로 쓰지 말고 prompt/template부터 확인하세요.


In [14]:
@torch.inference_mode()
def score_df(df):
    rows = []
    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"score {EXPERIMENT}"):
        inputs = processor.apply_chat_template(
            build_messages(row), tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt",
            downsample_mode=CFG["downsample_mode"], max_slice_nums=36,
        ).to(MODEL_DEVICE)

        with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE):
            out = model(**inputs, downsample_mode=CFG["downsample_mode"])

        last_idx = int(inputs["attention_mask"][0].sum().item()) - 1
        next_logits = out.logits[0, last_idx]
        s = next_logits.index_select(0, choice_ids).float()
        logp = torch.log_softmax(s, dim=0).cpu().numpy()
        pred_idx = int(np.argmax(logp))

        rec = {
            "row_idx": i,
            "path": str(row["path"]),
            "question": str(row["question"]),
            "gold": str(row["answer"]).strip().lower() if "answer" in row else None,
            "pred": CHOICES[pred_idx],
        }
        for c, v in zip(CHOICES, logp):
            rec[f"logp_{c}"] = float(v)
        rows.append(rec)
    return pd.DataFrame(rows)

valid_scores = score_df(valid_subset)
valid_acc = (valid_scores["pred"] == valid_scores["gold"]).mean()
VALID_SCORE_PATH = RUN_DIR / "valid_scores.csv"
valid_scores.to_csv(VALID_SCORE_PATH, index=False)
print(f"Validation accuracy: {valid_acc:.5f}")
print("saved:", VALID_SCORE_PATH)
valid_scores.head()


score A_backbone_16x: 100%|██████████| 31/31 [00:10<00:00,  3.07it/s]

Validation accuracy: 0.70968
saved: c:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\valid_scores.csv


,row_idx,path,question,gold,pred,logp_a,logp_b,logp_c,logp_d
0,0,train/train_3576.jpg,이 장난감 가격표에 따르면 'Woody Sheriff'의 가격은 얼마인가요?,d,d,-10.839909,-11.855534,-10.164127,-6.532456e-05
1,1,train/train_4232.jpg,사진 속 치킨팝 과자의 가격은 얼마인가요?,c,d,-13.062507,-15.031257,-12.250007,-7.152532e-06
2,2,train/train_6594.jpg,이 건물 유리창에 적힌 은행의 이름은 무엇인가요?,a,c,-2.183413,-6.808413,-0.120913,-8.964663e+00
3,3,train/train_6630.jpg,사진 속 상호명 중에서 24시간 운영되는 스터디 카페 이름은 무엇인가요?,d,d,-16.375000,-18.406250,-15.125000,-3.576278e-07
4,4,train/train_0535.jpg,이 도로 표지판에 따르면 '연주로'까지 거리는 얼마인가요?,c,c,-14.750003,-13.031253,-0.000003,-1.431250e+01


## 7. v4와 오답 overlap / ensemble


In [15]:
# baseline_v4_with_scores.ipynb를 먼저 실행하면 기본 위치에 생성됩니다.
V4_SCORE_PATH = OUTPUT_DIR / "v4_valid_scores.csv"

if V4_SCORE_PATH.exists():
    v4 = pd.read_csv(V4_SCORE_PATH)
    v5 = valid_scores.copy()
    key = ["path", "question"]
    m = v4.merge(v5, on=key, suffixes=("_v4", "_v5"), validate="one_to_one")

    v4_acc = (m.pred_v4 == m.gold_v4).mean()
    v5_acc = (m.pred_v5 == m.gold_v5).mean()
    both_wrong = ((m.pred_v4 != m.gold_v4) & (m.pred_v5 != m.gold_v5)).mean()
    v4_only_wrong = ((m.pred_v4 != m.gold_v4) & (m.pred_v5 == m.gold_v5)).mean()
    v5_only_wrong = ((m.pred_v4 == m.gold_v4) & (m.pred_v5 != m.gold_v5)).mean()

    # log-prob soft voting. alpha=0.5부터 보고 필요하면 validation에서만 탐색합니다.
    alphas = np.linspace(0, 1, 21)
    ens = []
    for alpha in alphas:
        mat = np.stack([
            alpha*m[f"logp_{c}_v4"].to_numpy() + (1-alpha)*m[f"logp_{c}_v5"].to_numpy()
            for c in CHOICES
        ], axis=1)
        pred = np.array(CHOICES)[mat.argmax(1)]
        acc = (pred == m.gold_v4.to_numpy()).mean()
        ens.append((alpha, acc))

    print(f"v4 acc       : {v4_acc:.5f}")
    print(f"v5 acc       : {v5_acc:.5f}")
    print(f"both wrong   : {both_wrong:.5f}")
    print(f"v4-only wrong: {v4_only_wrong:.5f}  <- v5가 구한 문제")
    print(f"v5-only wrong: {v5_only_wrong:.5f}  <- v4가 구한 문제")
    print("best ensemble(alpha=v4 weight):", max(ens, key=lambda x: x[1]))
else:
    print("v4 score 파일 없음:", V4_SCORE_PATH)
    print("같이 제공한 baseline_v4_with_scores.ipynb를 실행한 뒤 다시 이 셀을 실행하세요.")


v4 score 파일 없음: c:\SSAFY\AIChallenge\output\v4_valid_scores.csv
같이 제공한 baseline_v4_with_scores.ipynb를 실행한 뒤 다시 이 셀을 실행하세요.


## 8. Test score + submission


In [16]:
# test에는 answer가 없으므로 score_df를 약간 감싸서 사용합니다.
def score_test(df):
    temp = df.copy()
    if "answer" not in temp.columns:
        temp["answer"] = ""
    return score_df(temp)

RUN_TEST = True
if RUN_TEST:
    test_scores = score_test(test_df)
    test_scores.to_csv(RUN_DIR / "test_scores.csv", index=False)
    preds = test_scores["pred"].tolist()

    sample_path = DATA_DIR / "sample_submission.csv"
    if sample_path.exists():
        submission = pd.read_csv(sample_path)
        submission["answer"] = preds
    else:
        submission = pd.DataFrame({"id": test_df["id"], "answer": preds})

    sub_path = RUN_DIR / "submission.csv"
    submission.to_csv(sub_path, index=False)
    print("saved:", sub_path)
    display(submission.head())


score A_backbone_16x: 100%|██████████| 6714/6714 [36:11<00:00,  3.09it/s]

saved: c:\SSAFY\AIChallenge\output\v5_minicpm46_A_backbone_16x\submission.csv


,id,answer
0,test_0001.jpg,b
1,test_0002.jpg,c
2,test_0003.jpg,d
3,test_0004.jpg,d
4,test_0005.jpg,c


## 실험 실행 순서

1. `A_backbone_16x` — 가장 먼저. v4보다 얼마나 다르게 틀리는지 확인
2. `B_resolution_4x` — A에서 OCR/작은 글씨 오답이 많으면 실행
3. `C_projector_4x` — B에서도 visual alignment 문제가 남을 때 실행

판단 기준은 단일 accuracy만이 아닙니다.

- v5 accuracy
- `v4-only wrong`: v4는 틀렸는데 v5가 맞힌 비율
- `v5-only wrong`: v5는 틀렸는데 v4가 맞힌 비율
- validation soft-voting ensemble의 상승 여부

`C_projector_4x`는 A/B보다 실험 변수가 하나 더 많으므로 A/B 결과가 나온 뒤 진행하는 것을 권장합니다.
